## Exploring predictions

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import sys
import gc
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import silence_tensorflow

import experiment_settings
import build_model
import build_data
import plots
import methods
import predictions
import read_landsat

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
print(tf.config.list_physical_devices('GPU'))

python version = 3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:12:31) [Clang 14.0.6 ]
numpy version = 1.23.2
tensorflow version = 2.10.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# GET SETTINGS
EXP_NAME = "exp2"
REWRITE = False
settings = experiment_settings.get_settings(EXP_NAME)

directory_paths = methods.get_directories()
SAVE_MODEL_DIRECTORY = directory_paths["save_model_dir"]
DATA_DIRECTORY = directory_paths["data_dir"]
FIGURE_DIRECTORY = directory_paths["figures_dir"]
PREDICTIONS_DIRECTORY = directory_paths["predictions_dir"]
LANDSAT_DIRECTORY = directory_paths["landsat_dir"]

In [4]:
# GET THE DATA
import read_landsat
import methods
imp.reload(build_data)
imp.reload(read_landsat)
imp.reload(methods)

settings["batch_size"] = 128
settings["mode"] = "inference"
settings["inference_region"] = (-90, 90, -180, 180)
model = build_model.get_model(settings)

for year in (2020,):#np.arange(2019,2023):

    print(' --- ' + str(year) + '---')
    settings["inference_years"] = (year, )
    filenames_list = []
        
    (min_latfile, 
     max_latfile, 
     min_lonfile, 
     max_lonfile,) = read_landsat.get_landsat_bounds(settings, region=settings["inference_region"])

    for latfile in np.arange(min_latfile + settings["tile_len_deg"], 
                             max_latfile + settings["tile_len_deg"], 
                             settings["tile_len_deg"]):
        
        for lonfile in np.arange(min_lonfile, 
                                 max_lonfile, 
                                 settings["tile_len_deg"]):

            # CHECK IF LANDSAT FILE EXISTS
            settings["tile"] = (latfile - settings["tile_len_deg"], 
                                latfile, 
                                lonfile, 
                                lonfile + settings["tile_len_deg"])
            landsat_file = read_landsat.get_input_filename(settings["inference_years"], (latfile,), (lonfile,), settings)
            if os.path.isfile(LANDSAT_DIRECTORY + landsat_file[0] + ".tif"):
                print(landsat_file[0])
            else:
                # print("skipping missing file")
                continue

            # GET THE SAMPLE TAGS
            tags_inf, __ = build_data.get_tags(settings)
            if len(tags_inf[0]) == 0:
                continue
            
            # CHECK IF PREDICTION FILE ALREADY EXISTS
            predictions_filename = settings["exp_name"] + "_predictions_" + tags_inf[-1][0]
            filenames_list.append(predictions_filename + ".tif")
            if os.path.isfile(PREDICTIONS_DIRECTORY + predictions_filename + ".tif") and REWRITE is False:
                continue
            
            # BUILD THE DATA AND MAKE THE PREDICTIONS
            tfds_inf = build_data.build_tf_dataset(settings, tags_inf, settings["batch_size"])
            tfds_inf = tfds_inf.prefetch(tf.data.AUTOTUNE)

            # MAKE PREDICTIONS and SAVE AS TIF
            hfi_predict, hfi_labels, latlon_bounds = predictions.make_predictions(settings, model, tfds_inf, tags_inf)
            predictions.save_predictions_tif(hfi_predict, latlon_bounds, predictions_filename)
            filenames_list.append(predictions_filename + ".tif")

            # PLOT THE RESULTS
            lat0, lat1, lon0, lon1 = latlon_bounds

            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plots.plot_hfi_tile(hfi_predict, [lon0, lon1, lat1, lat0])
            plt.title('mlHFI Predictions for ' + str(settings["inference_years"][0]))
            plt.clim(0,100)

            plt.subplot(1,2,2)
            plots.plot_hfi_tile(hfi_labels, [lon0, lon1, lat1, lat0])
            plt.title('HFI Labels for ' + str(settings["inference_years"][0]))
            plt.clim(0,100)

            plt.savefig(FIGURE_DIRECTORY + predictions_filename + ".png")
            # plt.show()
            plt.close()

    # TILE THE PREDICTIONS TOGETHER
    mosaic, meta_data = predictions.create_mosaic(filenames_list)
    predictions.save_mosaic(settings, mosaic, meta_data)


Metal device set to: Apple M1 Max

systemMemory: 64.00 GB
maxCacheSize: 24.00 GB

 --- 2020---
output region shape = (372, 371)
n_inference = (138012,)
file = landsat_-7lat_105lon_2020
starting predictions ...
1079/1079 [==============================] - 83s 77ms/step
output region shape = (372, 371)
n_inference = (138012,)
file = landsat_-7lat_106lon_2020
starting predictions ...
1079/1079 [==============================] - 88s 81ms/step
output region shape = (372, 371)
n_inference = (138012,)
file = landsat_-7lat_107lon_2020
starting predictions ...
1079/1079 [==============================] - 92s 85ms/step
output region shape = (371, 371)
n_inference = (137641,)
file = landsat_-6lat_105lon_2020
starting predictions ...
1076/1076 [==============================] - 88s 82ms/step
output region shape = (371, 371)
n_inference = (137641,)
file = landsat_-6lat_106lon_2020
starting predictions ...
1076/1076 [==============================] - 97s 90ms/step
output region shape = (371, 371)
n_

In [ ]:
# imp.reload(predictions)
# imp.reload(experiment_settings)
# imp.reload(methods)
# settings = experiment_settings.get_settings(EXP_NAME)

# mosaic, meta_data = predictions.create_mosaic(filenames_list)
# predictions.save_mosaic(settings, mosaic, meta_data)